# Machine Translation with a Transformer

In this lab, we set up an English to French translation model:

1. use a pretrained **encoder-decoder** transformer, [MarianMT](https://huggingface.co/Helsinki-NLP/opus-mt-en-fr);
2. understand what happens during training (*teacher forcing*) and generation (beam search);
3. measure translation quality with the BLEU score, and what beam search brings for what it costs;
4. fine-tune the model on a corpus of novel translations, with a PyTorch training loop.

*Remember to enable the GPU: Runtime > Change runtime type > GPU.*

In [ ]:
!pip install -q datasets sacrebleu sentencepiece sacremoses

In [ ]:
import random
import time

import sacrebleu
import torch
import tqdm.auto
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Computing on: {device}")
torch.manual_seed(0)
random.seed(0)

## Loading the model

The University of Helsinki's `opus-mt` models are encoder-decoder transformers trained on the [OPUS](https://opus.nlpl.eu/) parallel corpora, one model per language pair.

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

Inspect the model (`print(model.config)`, `print(model)`):

- how many layers do the encoder and the decoder have?
- what are the model dimension and the number of attention heads?
- where is the cross-attention? What is it for?
- what is the vocabulary size? What kind of tokenization is used?

In [ ]:
# Your code here

### Solution

In [ ]:
print(model.config)
print(model.model.decoder.layers[0])
print(sum(p.numel() for p in model.parameters()) / 1e6, "million parameters")
print(tokenizer.tokenize("The neighbour's cat is extraordinarily lazy."))

- The encoder and the decoder each have 6 layers (`encoder_layers`, `decoder_layers`), of dimension 512 (`d_model`) with 8 attention heads.
- Each decoder layer contains a **causal** self-attention (`self_attn`) then a cross-attention (`encoder_attn`): queries come from the decoder, keys and values from the encoder's output. This is how the decoder "reads" the source sentence for every generated word.
- The vocabulary has about 60,000 subwords, obtained with SentencePiece (`▁` marks the start of a word).

## Translating

The `generate` method implements the decoding loop. By default, this model uses **beam search**: instead of only keeping the most probable word at each step (greedy decoding), it keeps the `num_beams` most probable translation beginnings and picks the best complete translation at the end.

In [ ]:
def translate(
  sentences: list[str], num_beams: int = 4, batch_size: int = 32
) -> list[str]:
  model.eval()
  translations = []
  for i in range(0, len(sentences), batch_size):
    batch = tokenizer(
      sentences[i : i + batch_size],
      return_tensors="pt",
      padding=True,
      truncation=True,
      max_length=128,
    ).to(device)
    with torch.no_grad():
      generated = model.generate(**batch, num_beams=num_beams, max_length=256)
    translations += tokenizer.batch_decode(generated, skip_special_tokens=True)
  return translations


examples = [
  "I think, therefore I am.",
  "The weather is lovely today, let's go for a walk.",
  "Could you please send me the report before Friday?",
]
for source, target in zip(examples, translate(examples)):
  print(f"{source}\n  → {target}")

## The data: novel translations

We use [OPUS Books](https://huggingface.co/datasets/Helsinki-NLP/opus_books): sentences from public domain novels aligned with their French translation. The literary style (dialogues, old-fashioned turns of phrase) differs from the texts the model was mostly trained on.

OPUS Books is part of OPUS, though, and OPUS-MT trains each model on nearly all the OPUS corpora of its language pair: our test sentences may have been seen during pretraining, which would flatter the model's score before fine-tuning.

About one French sentence in five in this corpus uses the typographic apostrophe (`’`), which the model's vocabulary lacks: it would become `<unk>`, which `skip_special_tokens=True` erases when decoding, and a fine-tuned model would write "simposait" for "s’imposait". The BLEU score, for its part, would count as an error each straight apostrophe of the model against a typographic one in the reference. So we replace it with the straight apostrophe as soon as we read the data.

We keep 1,000 pairs for testing, 1,000 for validation (to check for overfitting during fine-tuning) and 20,000 for training (so that the lab stays fast).

In [ ]:
books = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")


def straighten_apostrophes(example: dict) -> dict:
  # The model's vocabulary only has the straight apostrophe
  pair = example["translation"]
  return {"translation": {k: text.replace("’", "'") for k, text in pair.items()}}


books = books.map(straighten_apostrophes).shuffle(seed=0)
test_pairs = books.select(range(1000))
val_pairs = books.select(range(1000, 2000))
train_pairs = books.select(range(2000, 22000))
for pair in test_pairs.select(range(3))["translation"]:
  print(pair)

## Measuring quality: the BLEU score

The [BLEU](https://en.wikipedia.org/wiki/BLEU) score compares the *n*-grams (sequences of 1 to 4 words) of a machine translation with those of a reference translation. It ranges from 0 to 100. The thresholds often quoted (above 30, understandable translations; above 50, good quality) are only an order of magnitude: the score depends on the language, the corpus, the tokenization and how freely the reference translations are worded. Novel translators take many liberties: correct translations get a lower score there than on technical texts.

It is an imperfect measure (a correct translation phrased differently is penalized), but it makes it possible to compare models on the same corpus. chrF (`sacrebleu.corpus_chrf`), which counts character *n*-grams, is more tolerant of variations in form; COMET, a learned model, judges closer to a human.

Write a function `bleu(pairs, num_beams)` that translates the English sentences of `pairs` with `translate` and returns the BLEU score of these translations, computed with `sacrebleu.corpus_bleu`. Compute `bleu_before`, the pretrained model's score on the test set.

In [ ]:
def bleu(pairs, num_beams: int = 4) -> float:
  # Your code here
  ...

### Additional hints

The calls and parameters to use:

- `pairs["translation"]`: the list of pairs, each one a dictionary `{"en": ..., "fr": ...}`
- `translate(sources, num_beams=num_beams)`: the translations, in the order of the sources
- `sacrebleu.corpus_bleu(translations, [references])`: the list of translations, then a list of reference series (one series per translator, here only one); the score is in the `score` attribute
- The pretrained model gets about 23 on this test set

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def bleu(pairs, num_beams: int = 4) -> float:
  sources = [pair["en"] for pair in pairs["translation"]]
  references = [pair[...] for pair in ...]
  return sacrebleu.corpus_bleu(translate(..., num_beams=...), [...]).score


bleu_before = bleu(...)
print(f"BLEU before fine-tuning: {bleu_before:.1f}")
```

### Solution

In [ ]:
def bleu(pairs, num_beams: int = 4) -> float:
  sources = [pair["en"] for pair in pairs["translation"]]
  references = [pair["fr"] for pair in pairs["translation"]]
  return sacrebleu.corpus_bleu(
    translate(sources, num_beams=num_beams), [references]
  ).score


bleu_before = bleu(test_pairs)
print(f"BLEU before fine-tuning: {bleu_before:.1f}")

## Greedy or beam?

Beam search finds more probable translations than greedy decoding, but it follows `num_beams` translation beginnings at once. Does it gain enough quality for what it costs?

Measure the BLEU score and the translation time of the first 200 test sentences (`test_pairs.select(range(200))`) with `num_beams=1` (greedy decoding) and `num_beams=4`, timing each call of `bleu` with `time.perf_counter`. Conclude.

**Useful documentation**:

- https://docs.python.org/3/library/time.html#time.perf_counter

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `subset = test_pairs.select(range(200))`: the first 200 pairs, a `Dataset` like `test_pairs`
- `start = time.perf_counter()` before the call, `time.perf_counter() - start` after: the duration in seconds
- `bleu(subset, num_beams=num_beams)` in a loop over `(1, 4)`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
subset = test_pairs.select(range(200))
for num_beams in (1, 4):
  start = time.perf_counter()
  score = bleu(..., num_beams=...)
  elapsed = ... - start
  print(f"num_beams={num_beams}: BLEU {score:.1f} in {elapsed:.1f} s")
```

### Solution

In [ ]:
subset = test_pairs.select(range(200))
for num_beams in (1, 4):
  start = time.perf_counter()
  score = bleu(subset, num_beams=num_beams)
  elapsed = time.perf_counter() - start
  print(f"num_beams={num_beams}: BLEU {score:.1f} in {elapsed:.1f} s")

On these 200 sentences, beam search gains about one BLEU point (23.7 against 22.8 for greedy decoding), and the translation time roughly doubles on a GPU (7.5 s against 3.5 s), against about 60% more on a CPU; the gap depends on the batch size. Beam search is only an approximate search for the most probable translation, wider than greedy decoding: it finds better ones, but the gain is modest. The choice depends on the use: greedy decoding to translate a lot of text fast, beam search when each translation matters (it is this model's default setting, `num_beams=4`).

## Under the hood: *teacher forcing*

During training, we don't generate word by word: the decoder is given the **reference translation** shifted by one position, and asked to predict each next word. Thanks to the causal mask, all positions are predicted in a single pass.

The tokenizer prepares both sides: `text_target` produces the `labels`, and the model computes the cross-entropy loss itself when given these `labels`.

Compute the model's loss on the sentence pair of the next cell, without computing any gradient. Print the shape of the `logits`: what does each dimension correspond to?

**Useful documentation**:

- https://huggingface.co/docs/transformers/main_classes/output#transformers.modeling_outputs.Seq2SeqLMOutput

In [ ]:
pair = train_pairs[0]["translation"]
batch = tokenizer(pair["en"], text_target=pair["fr"], return_tensors="pt").to(device)
print(batch.keys())
# Your code here

### Additional hints

The calls and parameters to use:

- `torch.no_grad()`: no gradient for a mere measurement
- `output = model(**batch)`: with the batch's `labels`, `output.loss` is the mean cross-entropy over the translation's tokens
- `output.logits.shape`: three dimensions, the last one being 59514
- `tokenizer.convert_ids_to_tokens(batch["labels"][0])`: the subwords of the reference translation, to count the positions

A skeleton to copy into the exercise cell, replacing each `...`:

```python
with torch.no_grad():
  output = model(...)
print(f"Loss: {output.loss.item():.3f}")
print(f"Logits shape: {tuple(...)}")
print(tokenizer.convert_ids_to_tokens(batch["labels"][0]))
```

### Solution

In [ ]:
with torch.no_grad():
  output = model(**batch)
print(f"Loss: {output.loss.item():.3f}")
print(f"Logits shape: {tuple(output.logits.shape)}")
print(tokenizer.convert_ids_to_tokens(batch["labels"][0]))

The logits have shape `(batch, translation length, vocabulary size)`: for each position of the reference translation, one score per vocabulary subword.

## Preparing the training data

Write a function `preprocess(examples)` that tokenizes a batch of dataset examples (English sources and French targets, truncated to 128 tokens) with `tokenizer(..., text_target=...)`, then apply it to `train_pairs` with `Dataset.map` (`batched=True`) to get `train_tokenized`, without keeping the original text column.

The provided `DataCollatorForSeq2Seq` then pads the sequences of a batch. Why does it replace the padding of the `labels` with `-100`?

**Useful documentation**:

- https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.map

In [ ]:
def preprocess(examples: dict) -> dict:
  # Your code here
  ...

### Additional hints

The calls and parameters to use:

- `examples["translation"]`: with `batched=True`, the list of a batch's pairs, each one `{"en": ..., "fr": ...}`
- `tokenizer(sources, text_target=targets, max_length=128, truncation=True)`: `input_ids` and `attention_mask` of the sources, `labels` of the targets; no padding here, the collator does it batch by batch
- `remove_columns=train_pairs.column_names`: removes the `translation` column, text the collator couldn't assemble into tensors
- `train_tokenized` has 20,000 rows and the columns `input_ids`, `attention_mask` and `labels`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def preprocess(examples: dict) -> dict:
  sources = [pair["en"] for pair in examples[...]]
  targets = [... for pair in examples["translation"]]
  return tokenizer(sources, text_target=..., max_length=..., truncation=True)


train_tokenized = train_pairs.map(preprocess, batched=True, remove_columns=...)
print(train_tokenized)
```

### Solution

In [ ]:
def preprocess(examples: dict) -> dict:
  sources = [pair["en"] for pair in examples["translation"]]
  targets = [pair["fr"] for pair in examples["translation"]]
  return tokenizer(sources, text_target=targets, max_length=128, truncation=True)


train_tokenized = train_pairs.map(
  preprocess, batched=True, remove_columns=train_pairs.column_names
)
print(train_tokenized)

`-100` is the index `cross_entropy` ignores (`ignore_index`): padding positions don't count in the loss.

### The batches

The collator assembles the batches of `train_loader`, and those of `val_loader`, which will measure the validation loss at the end of fine-tuning.

In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer, model=model)
train_loader = DataLoader(
  train_tokenized, batch_size=32, shuffle=True, collate_fn=collator
)
val_tokenized = val_pairs.map(
  preprocess, batched=True, remove_columns=val_pairs.column_names
)
val_loader = DataLoader(val_tokenized, batch_size=64, collate_fn=collator)
batch = next(iter(train_loader))
print({key: value.shape for key, value in batch.items()})

## Fine-tuning

Before fine-tuning the model, let's keep its translations of five test sentences: the last part will compare them with those of the fine-tuned model.

In [ ]:
samples = test_pairs.select(range(5))["translation"]
before = translate([pair["en"] for pair in samples])

Write the training loop for one epoch:

- `AdamW` optimizer, learning rate of `5e-5` (small: we are adjusting an already good model);
- a linear learning rate schedule with *warmup* (`get_linear_schedule_with_warmup`, 10% of the steps as warmup);
- the loss is `model(**batch).loss`.

At the end of the epoch, compute the mean loss over `val_loader`, in evaluation mode and without gradient. The pretrained model's is about 2.1: does fine-tuning lower it? A widening gap with the training loss would signal overfitting.

On a Colab GPU, one epoch takes a few minutes.

**Useful documentation**:

- https://huggingface.co/docs/transformers/main_classes/optimizer_schedules#transformers.get_linear_schedule_with_warmup

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `torch.optim.AdamW(model.parameters(), lr=5e-5)`
- `n_steps = len(train_loader)`: the number of steps of one epoch; `get_linear_schedule_with_warmup(optimizer, num_warmup_steps=n_steps // 10, num_training_steps=n_steps)`
- `scheduler.step()` after each `optimizer.step()`: the schedule moves one step per batch, not per epoch
- `{key: value.to(device) for key, value in batch.items()}`: the batch on the GPU
- `model.train()` before the loop (`translate` put the model in evaluation mode), `model.eval()` and `torch.no_grad()` for the validation
- the validation loss: the mean of `model(**batch).loss.item()` over the batches of `val_loader`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(model.parameters(), lr=...)
n_steps = len(train_loader)
scheduler = get_linear_schedule_with_warmup(
  optimizer, num_warmup_steps=..., num_training_steps=...
)

model.train()
running_loss = 0.0
for step, batch in enumerate(tqdm.auto.tqdm(train_loader), 1):
  batch = {key: value.to(device) for key, value in batch.items()}
  optimizer.zero_grad()
  loss = model(**batch).loss
  ...
  optimizer.step()
  ...
  running_loss += loss.item()
  if step % 100 == 0:
    print(f"Step {step}: loss {running_loss / 100:.3f}")
    running_loss = 0.0

model.eval()
val_loss = 0.0
with torch.no_grad():
  for batch in val_loader:
    batch = {key: value.to(device) for key, value in batch.items()}
    val_loss += ...
print(f"Validation loss: {val_loss / len(val_loader):.3f}")
```

### Solution

In [ ]:
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
n_steps = len(train_loader)
scheduler = get_linear_schedule_with_warmup(
  optimizer, num_warmup_steps=n_steps // 10, num_training_steps=n_steps
)

model.train()
running_loss = 0.0
for step, batch in enumerate(tqdm.auto.tqdm(train_loader), 1):
  batch = {key: value.to(device) for key, value in batch.items()}
  optimizer.zero_grad()
  loss = model(**batch).loss
  loss.backward()
  optimizer.step()
  scheduler.step()
  running_loss += loss.item()
  if step % 100 == 0:
    print(f"Step {step}: loss {running_loss / 100:.3f}")
    running_loss = 0.0

model.eval()
val_loss = 0.0
with torch.no_grad():
  for batch in val_loader:
    batch = {key: value.to(device) for key, value in batch.items()}
    val_loss += model(**batch).loss.item()
print(f"Validation loss: {val_loss / len(val_loader):.3f}")

## Evaluation after fine-tuning

Compute `bleu_after`, the fine-tuned model's BLEU score on the test set, and compare it with `bleu_before`. Translate the sentences of `samples` again, and print for each one the source, the reference, the translation before fine-tuning (`before`) and the one after. What do you notice?

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `bleu(test_pairs)`: the same computation as before fine-tuning
- `after = translate([pair["en"] for pair in samples])`: the new translations, in the order of `before`
- `zip(samples, before, after)`: each pair, its translation before and the one after
- Before fine-tuning, the score was about 23

A skeleton to copy into the exercise cell, replacing each `...`:

```python
bleu_after = bleu(...)
print(f"BLEU before: {bleu_before:.1f}, after: {bleu_after:.1f}")

after = translate([pair["en"] for pair in ...])
for pair, old, new in zip(samples, before, after):
  print(f"Source    : {pair['en']}")
  print(f"Reference : {pair['fr']}")
  print(f"Before    : {...}")
  print(f"After     : {...}")
  print("-" * 80)
```

### Solution

In [ ]:
bleu_after = bleu(test_pairs)
print(f"BLEU before: {bleu_before:.1f}, after: {bleu_after:.1f}")

after = translate([pair["en"] for pair in samples])
for pair, old, new in zip(samples, before, after):
  print(f"Source    : {pair['en']}")
  print(f"Reference : {pair['fr']}")
  print(f"Before    : {old}")
  print(f"After     : {new}")
  print("-" * 80)

The BLEU score generally improves from the first epoch on (from 22.8 to 26.8 in our run), and the validation loss goes down: the model adapts to the style and vocabulary of novels. The fine-tuned translations also follow the reference's phrasing more closely, though not all are better: in our run, one example's "journeyman", "compagnon" before fine-tuning as in the reference, becomes "voyageur". This is transfer learning applied to text: start from a general-purpose model and adjust it with a little data from the target domain.

To go further:

- train for several epochs, measuring the validation loss at the end of each one: when it stops decreasing while the training loss still goes down, the model is overfitting;
- also measure chrF (`sacrebleu.corpus_chrf`) before and after fine-tuning: do both measures show the same progress?
- save the fine-tuned model with `model.save_pretrained("opus-mt-en-fr-books")` and `tokenizer.save_pretrained(...)`, then reload it with `from_pretrained`.